# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [3]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")
 

OpenAI client ready - using model gpt-4o-mini


In [4]:
def ask(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 1200, ai_model: str="gpt-4o-mini"):
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [5]:
prompt = "What is the capital of France?"
result = ask(prompt)
display(Markdown(f"**Q**: {prompt}\n\n**A**: {result}"))

**Q**: What is the capital of France?

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [6]:
prompt = """Extract the email address from this text:

Hi, you can reach me at john.doe@example.com for more information."""

result = ask(prompt, max_resp_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `The email address extracted from the text is: john.doe@example.com.`

In [7]:
prompt = """Extract the email address from this text and return ONLY the email, nothing else:

Hi, you can reach me at john.doe@example.com for more information."""

result = ask(prompt, max_resp_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [8]:
prompt = "Explain what an API is to a non-technical person."
system= "You are a professional technical writer who explains complex concepts simply."

result = ask(prompt, system_content= system)

display(Markdown(f"### Technical Writer\n\n{result}"))

### Technical Writer

Sure! An API, or Application Programming Interface, is like a waiter at a restaurant. When you go to a restaurant, you look at the menu, choose what you want, and then the waiter takes your order to the kitchen and brings your food back to you.

In the same way, an API is a set of rules that allows different software applications to communicate with each other. It acts as a middleman between different pieces of software, letting them share information and work together without needing to know the details of how each one is built.

For example, when you use an app on your phone to check the weather, that app uses an API to ask another service for the weather data. The API sends your request to the weather service, gets the information you need, and then sends it back to the app, which displays it for you to see.

So, in short, an API helps different software talk to each other, making it easier for us to use various services and features without needing to understand all the technical details behind them.

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [10]:
texts = [
    "I absolutely love this product! Best purchase ever",
    "The shipping was delayed and the item arrived damaged",
    "It's okay, nothing special but does the job"
]

print(f"{'Text':<55} {'Sentiment'}")
print("_" * 101)

for text in texts:
    prompt = f"Classify the sentiment as exactly one word: positive, negative or neutral\n\nText: {text}\n\nClassification:"
    sentiment = ask(prompt, temperature=0.0, max_resp_tokens=10).strip().lower()
    emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
    print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
_____________________________________________________________________________________________________
I absolutely love this product! Best purchase ever...   🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but does the job...          🟡 neutral


# Role - Context -Structure

In [11]:
prompt = "Explain what an API is to 10 years old child. Do it in 3 short sentences."
system = "You are my school teacher who loves analogies and explain everything in easy way"

result = ask(prompt, system_content=system)

display(Markdown(f"### Technical writer:\n\n{result}"))

### Technical writer:

An API is like a waiter in a restaurant. When you want food, you tell the waiter what you want, and they bring it to you from the kitchen. In the same way, an API helps different computer programs talk to each other and share information easily!

In [12]:
prompt = "Explain what an API is to 10 years old child. Do it in 3 short sentences. Put them in ordered bullets"
system = "You are my school teacher who loves analogies and explain everything in easy way"

result = ask(prompt, system_content=system)

display(Markdown(f"### Technical writer:\n\n{result}"))

### Technical writer:

Sure! Here’s a simple way to understand an API:

1. Imagine you go to a restaurant, and you tell the waiter what food you want from the menu.  
2. The waiter takes your order to the kitchen and brings your food back to you.  
3. An API is like that waiter; it helps different computer programs talk to each other and get what they need!

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [13]:
import json

prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

JSON:
"""

result = ask(prompt, temperature=0.0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

Raw output: ```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```
Not valid JSON


### Below a better prompt to give only the Raw JSON

In [14]:
prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON objects, nothing else.

JSON:
"""

result = ask(prompt, temperature=0.0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [15]:
prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

JSON:
"""

response_json = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages=[{'role': 'user', 'content': prompt}],
    response_format={"type": "json_object"}, # force valid JSON output
    temperature= 0.0
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [16]:
prompt = "Return a short greeting and a lucky number."
response_json = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages=[{'role': 'user', 'content': prompt}],
    response_format={"type": "json_schema",
                     'json_schema': {
                         'name':"greeting_response",
                         'schema': {
                             "type": "object",
                             "properties": {
                                 "greeting": {"type": "string"},
                                 "luckyNumber": {"type": "integer"}
                             },
                             "required": ["greeting", "luckyNumber"],
                             "additionalProperties": False
                         },
                         "strict": True
                     }}, 
    temperature= 0.0
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "greeting": "Hello! Wishing you a wonderful day!",
  "luckyNumber": 7
}
```

In [17]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "yourpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples